In [ ]:
import wandb
from tqdm.auto import tqdm
import numpy as np

from monai.apps import DecathlonDataset
from monai.data import DataLoader, decollate_batch
from monai.losses import DiceLoss
from monai.config import print_config
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.networks.nets import SegResNet
import monai.transforms
from monai.utils import set_determinism

import torch
print_config()

In [ ]:
import os
os.environ['WANDB_API_KEY'] = 'aa5821234bd41893b5da6713a5d8f3da4815ce68'

wandb.login()

In [ ]:
wandb.init(project="monai-brain-tumor-segmentation")

In [ ]:
config = wandb.config
config.seed = 0
config.roi_size = [224, 224, 144]
config.batch_size = 1
config.num_workers = 4
config.max_train_images_visualized = 3
config.max_val_images_visualized = 3
config.dice_loss_smoothen_numerator = 0
config.dice_loss_smoothen_denominator = 1e-5
config.dice_loss_squared_prediction = True
config.dice_loss_target_onehot = False
config.dice_loss_apply_sigmoid = True
config.initial_learning_rate = 1e-4
config.weight_decay = 1e-5
config.max_train_epochs = 50
config.validation_intervals = 1
config.dataset_dir = "./dataset/"
config.checkpoint_dir = "./checkpoints"
config.inference_roi_size = (128, 128, 64)
config.max_prediction_images_visualized = 20

In [ ]:
from typing import Optional, List
import monai
import monai.transforms
import SimpleITK as sitk
import numpy as np 
import torch
from lightning import LightningDataModule
from torch.utils.data import ConcatDataset, DataLoader, Dataset, random_split
from torchvision.transforms import transforms
import json
import os
import rootutils

rootutils.setup_root(search_from="/work/hpc/spine-segmentation/notebooks/logger_wandb.ipynb", indicator="setup.py", pythonpath=True)
from src.data.transforms import array

# always starting with vanilla dataset, like its a norm to me now
class SpiderDataset(Dataset):
    num_class = 15
    def __init__(self, 
                 data = None,
                 data_dir: str = "", 
                 json_path: str = "",
                 ):
        super().__init__()
        self.data = list()
        self.data_dir = data_dir
        if data is not None:
            self.data = data
        else:
            if data_dir == "" or json_path == "":
                raise AssertionError("No dataset ?")
            self.setup(json_path)
    
    def setup(self, json_path):
        json_object = json.load(open(json_path, "r"))
        keys = json_object.keys()
        if "training" in keys:
            for key in keys:
                self.data.extend(json_object[key])
        else:
            try:
                self.data.extend(json_object)
            except:
                raise InsertionError("Something wrong with json file, cannot load or do anything, at all")
    
    def get_item(self, index: int):
        output = dict()
        output["image"] = ""
        output["label"] = ""
        if isinstance(self.data[index]["image"], list):
            output["image"] = [os.path.join(self.data_dir, image) for image in self.data[index]["image"]]
        else:
            # Add for debugging
            path = os.path.join(self.data_dir, self.data[index]["image"])
            output["image"] = path ##Set to 4 because the model_inferr in validation_step
        output["label"] = os.path.join(self.data_dir, self.data[index]["label"])
        return output
    
    # In case they query in list of index
    def __getitem__(self, index):
        output = None
        if not isinstance(index, int):
            output = []
            for id in index:
                output.append(self.get_item(id))
        else:
            output = self.get_item(int(index))
        
        return output
    
    def __len__(self) -> int:
        return len(self.data)
    
    

class SpiderTransformedDataset(Dataset):
    def __init__(self, 
                 dataset: SpiderDataset,
                 transform: monai.transforms.Compose):
        super().__init__()
        self.dataset = dataset
        self.transform = transform
    
    def __getitem__(self, index):
        output = None
        
        if not isinstance(index, int):
            output = []
            for id in index:
                output.append(self.transform(self.dataset[id]))
        else:
            output = self.transform(self.dataset[int(index)])

        return output
    
    def __len__(self) -> int:
        return len(self.dataset)

In [ ]:
dataset = SpiderDataset(data_dir = "/data/hpc/spine/dataset/spine_nii", json_path="/data/hpc/spine/jsons/test.json")
# dataset = SpiderDataset(data_dir = "./data/dataset", json_path="/data/hpc/spine/jsons/brats21_folds.json")

transform = monai.transforms.Compose([monai.transforms.LoadImaged(keys=["image", "label"], image_only = False),
                                        array.ConvertToMultiChannelBasedOnSpiderClassesdSemantic(keys=["label"]),
                                        monai.transforms.EnsureChannelFirstd(keys=["image"]),
                                        # monai.transforms.Spacingd(
                                        #     keys=["image", "label"],
                                        #     pixdim=(2.0, 2.0, 2.0),
                                        #     mode=("bilinear", "nearest"),
                                        # ),
                                    #   monai.transforms.ConvertToMultiChannelBasedOnBratsClassesd(keys=["label"]),
                                        # monai.transforms.Resized(keys=["image", "label"], spatial_size=(32, 420, 420)),
                                        monai.transforms.ToTensord(keys=["image", "label"]),])

transformed = SpiderTransformedDataset(dataset, transform)
id = 0
data = dataset[id]
images = transformed[id]
print(data)
print(images["image"].size(), images["image"].dtype)
print(images["label"].size(), images["label"].dtype)
print(images.keys())
print("------------------")
print(images['image_meta_dict'])
print("------------------")
print(images['label_meta_dict'])
print(images["label"].size())

# res = 0
# for image in transformed:
#     res = max(res, image["label"].size(0))

# print(res)
# [[  -3.3211,    0.0000,    0.0000,   66.6507],
#         [   0.0000,   -0.6250,    0.0000,   85.9260],
#         [   0.0000,    0.0000,    0.5006, -119.7997],
#         [   0.0000,    0.0000,    0.0000,    1.0000]]

## Log wandb

In [ ]:
def log_data_samples_into_tables(
    sample_image: np.array,
    sample_label: np.array,
    split: str = None,
    data_idx: int = None,
    table: wandb.Table = None,
):
    print(sample_image.shape)
    num_channels, num_slices, _, _ = sample_image.shape
    with tqdm(total=num_slices, leave=False) as progress_bar:
        for slice_idx in range(num_slices):
            ground_truth_wandb_images = []
            for channel_idx in range(num_channels):
                ground_truth_wandb_images.append(
                    wandb.Image(
                        sample_image[channel_idx, slice_idx, :, :],
                        masks={
                            "Pred: Vertebral": {    
                                "mask_data": sample_label[0, slice_idx, :, :],
                                "class_labels": {1: "Pred: Vertebral"},
                            },
                            "GT: Vertebral": {    
                                "mask_data": sample_label[0, slice_idx, :, :] * 2,
                                "class_labels": {2: "GT: Vertebral"},
                            },

                            "Pred: Canal": {
                                "mask_data": sample_label[1, slice_idx, :, :] * 3,
                                "class_labels": {3: "Pred: Canal"},
                            },
                            "GT: Canal": {
                                "mask_data": sample_label[1, slice_idx, :, :] * 4,
                                "class_labels": {4: "GT: Canal"},
                            },
                            
                            "Pred: Disk": {
                                "mask_data": sample_label[2, slice_idx, :, :] * 5,
                                "class_labels": {5: "Pred: Disk"},
                            },
                            "GT: Disk": {
                                "mask_data": sample_label[2, slice_idx, :, :] * 6,
                                "class_labels": {6: "GT: Disk"},
                            },
                        },
                    )
                )
            table.add_data(split, data_idx, slice_idx, *ground_truth_wandb_images)
            progress_bar.update(1)
    return table


table = wandb.Table(
    columns=[
        "Split",
        "Data Index",
        "Slice Index",
        "Image-Channel"
    ]
)

In [ ]:
# Generate visualizations for train_dataset
max_samples = (
    min(config.max_train_images_visualized, len(transformed))
    if config.max_train_images_visualized > 0
    else len(transformed)
)
progress_bar = tqdm(
    enumerate(transformed.__getitem__([0, 1, 2])),
    total=max_samples,
    desc="Generating Train Dataset Visualizations:",
)
for data_idx, sample in progress_bar:
    sample_image = sample["image"].detach().cpu().numpy()
    sample_label = sample["label"].detach().cpu().numpy()
    table = log_data_samples_into_tables(
        sample_image,
        sample_label,
        split="train",
        data_idx=data_idx,
        table=table,
    )

# # Generate visualizations for val_dataset
# max_samples = (
#     min(config.max_val_images_visualized, len(val_dataset))
#     if config.max_val_images_visualized > 0
#     else len(val_dataset)
# )
# progress_bar = tqdm(
#     enumerate(val_dataset[:max_samples]),
#     total=max_samples,
#     desc="Generating Validation Dataset Visualizations:",
# )
# for data_idx, sample in progress_bar:
#     sample_image = sample["image"].detach().cpu().numpy()
#     sample_label = sample["label"].detach().cpu().numpy()
#     table = log_data_samples_into_tables(
#         sample_image,
#         sample_label,
#         split="val",
#         data_idx=data_idx,
#         table=table,
#     )

# Log the table to your dashboard
wandb.log({"Tumor-Segmentation-Data": table})
     

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(transformed[1]["image"][0, 25, :, :], cmap="gray")

In [ ]:
bruh = monai.transforms.Compose([monai.transforms.LoadImage(image_only = False),
                                        # array.ConvertToMultiChannelBasedOnSpiderClassesdSemantic(keys=["label"]),
                                        # Spacing(
                                        # keys=["image", "label"],
                                        # pixdim=(1.0, 1.0, 1.0),
                                        # mode=("bilinear", "nearest"),
                                    # ),
                                    #   monai.transforms.Resize(keys=["image", "label"], spatial_size=(250, 250, 155)),
                                        monai.transforms.ToTensor(),])


print(bruh('/Users/tiendzung/Downloads/BraTS2021_00630/BraTS2021_00630_t2.nii.gz'))

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt

seg_out = np.zeros((100,200,3))

nib.save(nib.Nifti1Image(seg_out.astype(np.uint8), affine=None), "/Users/tiendzung/Project/spine-segmentation/notebooks/test.nii.gz")

In [ ]:
img = nib.load("/Users/tiendzung/Downloads/spine_nii/masks/105_t1.nii.gz")
header = img.header
print(img.header)
print(header.get_data_shape())

In [ ]:
print(img.affine)

In [ ]:
image_data = img.get_fdata()
print(image_data.shape)

In [ ]:
plt.imshow(image_data[10, :, :],)

In [ ]:
affine = img.affine

affine[0, 0] = affine[0, 0] * -1
affine[1, 1] = affine[1, 1] * -1

print(affine)

nib.save(nib.Nifti1Image(image_data.astype(np.uint8), affine=affine), "/Users/tiendzung/Project/spine-segmentation/notebooks/test.nii.gz")

In [ ]:
test = nib.load("/Users/tiendzung/Downloads/BraTS2021_00630/BraTS2021_00630_t2.nii.gz")
print(test.header)
print(test.header.get_data_shape())

In [ ]:
brats_img = test.get_fdata()

plt.imshow(brats_img[:, :, 100])

## Merge mask

In [ ]:
from matplotlib import pyplot as plt
os.environ['CUDA_VISIBLE_DEVICES'] = '2'


In [ ]:
print(images["label"].size())
print(images["label"][0])

bruh = torch.sum(images["label"], dim=3)
print(bruh.size())


In [ ]:
from matplotlib import pyplot as plt
from PIL import Image
print(images["label"][0][12].shape)
print(np.array(images["label"][0][12]).shape)


# plt.imshow(Image.fromarray(np.array(images["image"][0][12])))
plt.imshow(np.rot90(np.array(images["image"][0][12]), 1))
plt.imshow(images["image"][0][12])

In [ ]:
from scipy.ndimage import zoom
from matplotlib import cm
import cv2


coronal_view = zoom(torch.sum(images["label"], dim=2).permute(0, 2, 1), (1, 1, images["label"].shape[2]/images["label"].shape[1]/6))
coronal_view = coronal_view[:,::-1,:]
sagittal_view = torch.sum(images["label"], dim=1)

print(f"Coronal View shape: {coronal_view.shape}")
print(f"Sagittal View shape: {sagittal_view.shape}")
plt.imshow(coronal_view[0])

In [ ]:
from matplotlib import pyplot as plt
import rootutils
rootutils.setup_root("/work/hpc/spine-segmentation/notebooks/logger_wandb.ipynb", indicator=".project-root", pythonpath=True)

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

from functools import partial

import nibabel as nib
import numpy as np

from lightning import LightningModule
import torch
from torch.utils.data import DataLoader, Dataset
# from src.models.brats21_module import Brats21LitModule
from src.models.spider_semantic_module import SpiderLitModule
from monai.inferers import sliding_window_inference

from monai.transforms.utils import allow_missing_keys_mode
from scipy.ndimage import zoom
from matplotlib import cm
from matplotlib.colors import Normalize

class Inferer(object):
    def __init__(
        self,
        exp_name: str,
        checkpoint_path: str,
        data_name: str,
        data_loader: DataLoader,
        net: torch.nn.Module,
        model_inferer: sliding_window_inference,
        device
    ):
        self.output_directory = "/work/hpc/spine-segmentation/outputs/" + exp_name ##"./outputs/" + exp_name
        print(os.getcwd())
        if not os.path.exists(self.output_directory):
            os.makedirs(self.output_directory)
            
        self.data_name = data_name
        self.data_loader = data_loader
        net.to(device)
        self.net = net
        model = SpiderLitModule.load_from_checkpoint(checkpoint_path=checkpoint_path)
        self.model_inferer = partial(model_inferer, predictor=model)
      
    def feed(self):
        with torch.no_grad():
            for i, batch in enumerate(self.data_loader):
                image = batch["image"].cuda()
                label = batch["label"].cuda()
                print(batch.keys())
                print(image.size())
                affine = batch["image_meta_dict"]["original_affine"][0].numpy()
                original_size = batch["image_meta_dict"]["spatial_shape"][0]

                img_name = batch["image_meta_dict"]["filename_or_obj"][0].split("/")[-1]
                print("CCCCCCCC" + img_name)
                print("Inference on case {}".format(img_name))
                prob = torch.sigmoid(self.model_inferer(image)) ## this or that
                # prob = torch.sigmoid(self.net(image)) ## that
                seg = prob[0].detach().cpu().numpy()
                self.logger(seg)
                seg = zoom(seg, (1, float(original_size[0])/seg.shape[1], float(original_size[1])/seg.shape[2], float(original_size[2])/seg.shape[3]))
                print(seg.shape)
                seg = (seg > 0.5).astype(np.int8)
                print("Bruhhhh")
                print(seg.shape)
                seg_out = np.zeros((seg.shape[1], seg.shape[2], seg.shape[3]))
                seg_out[seg[1] == 1] = 2
                seg_out[seg[0] == 1] = 1
                seg_out[seg[2] == 1] = 4
                # resize_op = self.data_loader.dataset.transform
                # with allow_missing_keys_mode:
                #     print(resize_op.inverse({"label": seg_out}))

                print(seg_out.shape)
                # seg_out = np.resize(seg_out, original_size.numpy())
                nib.save(nib.Nifti1Image(seg_out.astype(np.uint8), affine), os.path.join(self.output_directory, img_name))
            print("Finished inference!")
        
        # for i, batch in enumerate(self.data_loader):
        #     print(batch['image'].size())
        #     if (i == 0):
        #         print(batch['image'][0])

    def merge_image(self, vertebral_img, disk_img, canal_img = 0):
        if isinstance(canal_img, int) == False:
            canal_img = np.stack([canal_img, np.zeros(canal_img.shape), canal_img], axis = -1)
            canal_img = canal_img / canal_img.max() / 2

        norm_vertebral = Normalize(vmin=vertebral_img.min(), vmax=vertebral_img.max())

        disk_img = np.stack([disk_img, disk_img, np.zeros(disk_img.shape)], axis = -1)
        disk_img = disk_img / disk_img.max()

        vertebral_img = cm.viridis(norm_vertebral(vertebral_img))[:, :, :3] + disk_img + canal_img

        return vertebral_img

    def logger(self, predict_seg):
        coronal_view = torch.sum(predict_seg, dim=2).permute(0, 2, 1)
        sagittal_view = torch.sum(predict_seg, dim=1)

        image1 = self.merge_image(coronal_view[0], coronal_view[2])
        image2 = self.merge_image(sagittal_view[0], sagittal_view[2], sagittal_view[1])
        image2 = np.rot90(image2, 1)

        print(image1.shape)
        print(image2.shape)

        image_all = np.concatenate((image1, image2), axis = 1)

        plt.imshow(image_all)
        plt.show()
        

            
import hydra
from omegaconf import OmegaConf, DictConfig

config_name = "infer_spider"

config_path = os.path.join(os.environ["PROJECT_ROOT"], "configs/app")

cfg = OmegaConf.load(os.path.join(config_path, config_name + ".yaml"))

# print(OmegaConf.to_yaml(cfg))
inferer: Inferer = hydra.utils.instantiate(cfg)
print(inferer)
# print(type(inferer))
inferer.feed()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.colors import Normalize

def merge_image(vertebral_img, disk_img, canal_img = 0):
    if isinstance(canal_img, int) == False:
        canal_img = np.stack([canal_img, np.zeros(canal_img.shape), canal_img], axis = -1)
        canal_img = canal_img / canal_img.max() / 2

    norm_vertebral = Normalize(vmin=vertebral_img.min(), vmax=vertebral_img.max())

    disk_img = np.stack([disk_img, disk_img, np.zeros(disk_img.shape)], axis = -1)
    disk_img = disk_img / disk_img.max()

    vertebral_img = cm.viridis(norm_vertebral(vertebral_img))[:, :, :3] + disk_img + canal_img

    return vertebral_img


fig, axs = plt.subplots(1, 2, layout = "constrained", figsize=(10, 5))

# axs[0].imshow(coronal_view[0])
# axs[0].axis('off')

axs[0].imshow(merge_image(coronal_view[0], coronal_view[2]))
axs[0].axis('off')

axs[1].imshow(np.rot90(merge_image(sagittal_view[0], sagittal_view[2], sagittal_view[1]), 1))
axs[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
cv2.imwrite("/work/hpc/spine-segmentation/notebooks/test.png", coronal_view[2]) 

In [ ]:
a = np.array([[0 , 1], [2, 3]])

def test(b):
    b[1] = 60

test(a[0])

print(a[0])

In [ ]:
image1 = merge_image(coronal_view[0], coronal_view[2])
image2 = merge_image(sagittal_view[0], sagittal_view[2], sagittal_view[1])
image2 = np.rot90(image2, 1)
print(image1.shape)
print(image2.shape)

image_all = np.concatenate((image1, image2), axis = 1)

plt.imshow(image_all)

In [ ]:
from monai.data.utils import list_data_collate
dataloader = DataLoader(dataset=transformed, 
            batch_size=1,
            num_workers=4,
            pin_memory=False,
            shuffle=False,
            collate_fn = list_data_collate)

batch = next(iter(dataloader))

print(batch["label"][0].shape)

plt.imshow(batch["label"][0][0][10])

for i, batch in enumerate(dataloader):
    plt.imshow(batch["label"][0][0][10])
    plt.show()
        